# Column Transformer Practice with Titanic Dataset

## Two-Phase Approach: Manual → ColumnTransformer


## Phase 1: Manual Transformations (for understanding)


In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import warnings

warnings.filterwarnings("ignore")

In [2]:
# Load dataset
df = pd.read_csv(
    r"C:\Users\infor\OneDrive\Desktop\Machine Learning\DataSets\Titanic-Dataset.csv"
)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# Check data types and missing values
print("Data Info:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB
None

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked        

In [4]:
# Select relevant features for our practice
# Features: Age (numeric, missing), Pclass (numeric, no missing), Sex (categorical), Fare (numeric)
# Target: Survived

df_model = df[["Pclass", "Age", "Sex", "Fare", "Survived"]].copy()

# Check selected columns
print("Selected features info:")
print(df_model.info())
print("\nMissing values:")
print(df_model.isnull().sum())
print("\nValue counts for Sex:")
print(df_model["Sex"].value_counts())

Selected features info:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Age       714 non-null    float64
 2   Sex       891 non-null    str    
 3   Fare      891 non-null    float64
 4   Survived  891 non-null    int64  
dtypes: float64(2), int64(2), str(1)
memory usage: 39.0 KB
None

Missing values:
Pclass        0
Age         177
Sex           0
Fare          0
Survived      0
dtype: int64

Value counts for Sex:
Sex
male      577
female    314
Name: count, dtype: int64


In [5]:
# Remove rows with missing values for clean comparison
df_model = df_model.dropna()

# Separate features and target
X = df_model.drop(columns="Survived")
y = df_model["Survived"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("\nFeatures:")
print(X.head())

X shape: (714, 4)
y shape: (714,)

Features:
   Pclass   Age     Sex     Fare
0       3  22.0    male   7.2500
1       1  38.0  female  71.2833
2       3  26.0  female   7.9250
3       1  35.0  female  53.1000
4       3  35.0    male   8.0500


In [6]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (571, 4)
X_test shape: (143, 4)


### Manual Transformations (Understanding the Process)


In [7]:
# ⚠️ IMPORTANT: Fit transformers ONLY on training data
# Then apply to test data using transform() only (NOT fit_transform())

# Step 1: Impute missing values in Age
imputer = SimpleImputer(strategy="mean")
imputer.fit(X_train[["Age"]])  # Fit only on train data

X_train_age = imputer.transform(X_train[["Age"]])  # Transform train
X_test_age = imputer.transform(X_test[["Age"]])  # Transform test (NOT fit_transform)

print(f"Age after imputation (train): {X_train_age.shape}")
print(f"Age after imputation (test): {X_test_age.shape}")

Age after imputation (train): (571, 1)
Age after imputation (test): (143, 1)


In [8]:
# Step 2: One-hot encode Sex (categorical)
ohe = OneHotEncoder(sparse_output=False, drop="first")
ohe.fit(X_train[["Sex"]])  # Fit only on train data

X_train_sex = ohe.transform(X_train[["Sex"]])  # Transform train
X_test_sex = ohe.transform(X_test[["Sex"]])  # Transform test (NOT fit_transform)

print(f"Sex after encoding (train): {X_train_sex.shape}")
print(f"Sex after encoding (test): {X_test_sex.shape}")
print(f"Feature names: {ohe.get_feature_names_out(['Sex'])}")

Sex after encoding (train): (571, 1)
Sex after encoding (test): (143, 1)
Feature names: ['Sex_male']


In [9]:
# Step 3: Scale numeric features (Pclass, Fare)
scaler = StandardScaler()
scaler.fit(X_train[["Pclass", "Fare"]])  # Fit only on train data

X_train_numeric = scaler.transform(X_train[["Pclass", "Fare"]])  # Transform train
X_test_numeric = scaler.transform(
    X_test[["Pclass", "Fare"]]
)  # Transform test (NOT fit_transform)

print(f"Numeric features after scaling (train): {X_train_numeric.shape}")
print(f"Numeric features after scaling (test): {X_test_numeric.shape}")

Numeric features after scaling (train): (571, 2)
Numeric features after scaling (test): (143, 2)


In [10]:
# Step 4: Concatenate all transformed features in consistent order
# Order: [Pclass, Fare (scaled)] + [Sex (encoded)] + [Age (imputed)]

X_train_manual = np.concatenate((X_train_numeric, X_train_sex, X_train_age), axis=1)

X_test_manual = np.concatenate((X_test_numeric, X_test_sex, X_test_age), axis=1)

print(f"Combined X_train shape: {X_train_manual.shape}")
print(f"Combined X_test shape: {X_test_manual.shape}")
print(f"\nX_train_manual sample:\n{X_train_manual[:3]}")

Combined X_train shape: (571, 4)
Combined X_test shape: (143, 4)

X_train_manual sample:
[[ 0.92816778 -0.29381913  0.         31.        ]
 [ 0.92816778 -0.41638138  1.         26.        ]
 [ 0.92816778 -0.38315463  1.         30.        ]]


In [11]:
# Create column names for reference
manual_col_names = ["Pclass_scaled", "Fare_scaled", "Sex_male", "Age_imputed"]

X_train_manual_df = pd.DataFrame(X_train_manual, columns=manual_col_names)
X_test_manual_df = pd.DataFrame(X_test_manual, columns=manual_col_names)

print("X_train_manual_df:")
print(X_train_manual_df.head())

X_train_manual_df:
   Pclass_scaled  Fare_scaled  Sex_male  Age_imputed
0       0.928168    -0.293819       0.0         31.0
1       0.928168    -0.416381       1.0         26.0
2       0.928168    -0.383155       1.0         30.0
3       0.928168    -0.551227       1.0         33.0
4      -0.265489    -0.445740       1.0         25.0


## Phase 2: Using ColumnTransformer (Automated)


In [12]:
# Create ColumnTransformer with same transformations in SAME ORDER
ct = ColumnTransformer(
    transformers=[
        # Numeric features: scale Pclass and Fare
        ("scaler", StandardScaler(), ["Pclass", "Fare"]),
        # Categorical: one-hot encode Sex
        ("ohe", OneHotEncoder(sparse_output=False, drop="first"), ["Sex"]),
        # Numeric with missing: impute Age
        ("imputer", SimpleImputer(strategy="mean"), ["Age"]),
    ],
    remainder="drop",  # Drop any columns not specified
)

# ⚠️ IMPORTANT: Fit ONLY on training data
ct.fit(X_train)

# Transform both train and test using the fitted transformer
X_train_ct = ct.transform(X_train)  # Transform train
X_test_ct = ct.transform(X_test)  # Transform test (NOT fit_transform)

print(f"X_train (ColumnTransformer) shape: {X_train_ct.shape}")
print(f"X_test (ColumnTransformer) shape: {X_test_ct.shape}")
print(f"\nX_train_ct sample:\n{X_train_ct[:3]}")

X_train (ColumnTransformer) shape: (571, 4)
X_test (ColumnTransformer) shape: (143, 4)

X_train_ct sample:
[[ 0.92816778 -0.29381913  0.         31.        ]
 [ 0.92816778 -0.41638138  1.         26.        ]
 [ 0.92816778 -0.38315463  1.         30.        ]]


In [13]:
# Get feature names from ColumnTransformer
# Note: get_feature_names_out() requires sklearn 1.0+
try:
    ct_col_names = ct.get_feature_names_out()
    print(f"Feature names from ColumnTransformer: {ct_col_names}")
except:
    # Fallback for older sklearn versions
    ct_col_names = manual_col_names
    print(f"Using manual column names: {ct_col_names}")

X_train_ct_df = pd.DataFrame(X_train_ct, columns=ct_col_names)
X_test_ct_df = pd.DataFrame(X_test_ct, columns=ct_col_names)

print("\nX_train_ct_df:")
print(X_train_ct_df.head())

Feature names from ColumnTransformer: ['scaler__Pclass' 'scaler__Fare' 'ohe__Sex_male' 'imputer__Age']

X_train_ct_df:
   scaler__Pclass  scaler__Fare  ohe__Sex_male  imputer__Age
0        0.928168     -0.293819            0.0          31.0
1        0.928168     -0.416381            1.0          26.0
2        0.928168     -0.383155            1.0          30.0
3        0.928168     -0.551227            1.0          33.0
4       -0.265489     -0.445740            1.0          25.0


## Phase 3: Verify Equivalence


In [14]:
# Compare manual vs ColumnTransformer results
print("Comparing Manual vs ColumnTransformer Transformations:")
print("\n1. Training Data:")
print(f"   Manual shape: {X_train_manual.shape}")
print(f"   ColumnTransformer shape: {X_train_ct.shape}")
print(f"   Shapes match: {X_train_manual.shape == X_train_ct.shape}")

# Check if values are approximately equal (allowing for floating point precision)
train_values_match = np.allclose(X_train_manual, X_train_ct)
print(f"   Values match: {train_values_match}")

print("\n2. Test Data:")
print(f"   Manual shape: {X_test_manual.shape}")
print(f"   ColumnTransformer shape: {X_test_ct.shape}")
print(f"   Shapes match: {X_test_manual.shape == X_test_ct.shape}")

test_values_match = np.allclose(X_test_manual, X_test_ct)
print(f"   Values match: {test_values_match}")

print(
    "\n✅ SUCCESS!"
    if train_values_match and test_values_match
    else "\n❌ Values differ!"
)

Comparing Manual vs ColumnTransformer Transformations:

1. Training Data:
   Manual shape: (571, 4)
   ColumnTransformer shape: (571, 4)
   Shapes match: True
   Values match: True

2. Test Data:
   Manual shape: (143, 4)
   ColumnTransformer shape: (143, 4)
   Shapes match: True
   Values match: True

✅ SUCCESS!


## Phase 4: End-to-End Pipeline (ColumnTransformer + Model)


In [15]:
# Reload data for clean pipeline
X = df_model.drop(columns="Survived")
y = df_model["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create a complete pipeline
pipeline = Pipeline(
    [
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    ("scaler", StandardScaler(), ["Pclass", "Fare"]),
                    ("ohe", OneHotEncoder(sparse_output=False, drop="first"), ["Sex"]),
                    ("imputer", SimpleImputer(strategy="mean"), ["Age"]),
                ],
                remainder="drop",
            ),
        ),
        ("classifier", LogisticRegression(random_state=42, max_iter=1000)),
    ]
)

# Train pipeline (fit and transform happen automatically)
pipeline.fit(X_train, y_train)

print("Pipeline trained successfully!")

Pipeline trained successfully!


In [16]:
# Make predictions
y_pred_train = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)

# Evaluate
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

print("\nClassification Report (Test Set):")
print(
    classification_report(
        y_test, y_pred_test, target_names=["Did not survive", "Survived"]
    )
)

Training Accuracy: 0.8161
Test Accuracy: 0.7483

Classification Report (Test Set):
                 precision    recall  f1-score   support

Did not survive       0.80      0.78      0.79        87
       Survived       0.67      0.70      0.68        56

       accuracy                           0.75       143
      macro avg       0.74      0.74      0.74       143
   weighted avg       0.75      0.75      0.75       143



## Key Takeaways


### ✅ Best Practices for ColumnTransformer:

1. **Fit on Training Data Only**
   - `ct.fit(X_train)` - Fit parameters from training data
   - `ct.transform(X_test)` - Apply learned parameters to test data
   - ❌ Never: `ct.fit_transform(X_test)` - This causes data leakage!

2. **Consistent Column Ordering**
   - Transformer order in ColumnTransformer matches output order
   - Make sure column lists in transformers don't overlap

3. **Handle Missing Values**
   - Use SimpleImputer before scaling or encoding
   - Different strategies: 'mean', 'median', 'most_frequent'

4. **Scaling vs Encoding**
   - Scale numeric features (StandardScaler, MinMaxScaler)
   - Encode categorical features (OneHotEncoder, OrdinalEncoder)
   - Impute missing values (SimpleImputer)

5. **Pipeline Integration**
   - Combine ColumnTransformer + Estimator in Pipeline
   - Prevents accidental data leakage
   - Cleaner, reusable code

6. **Column Naming**
   - Use `get_feature_names_out()` to track output columns
   - Helpful for model interpretation

### ⚠️ Common Mistakes:

- Fitting on test data (data leakage)
- Using `fit_transform()` on test data
- Overlapping column lists across transformers
- Forgetting to handle missing values
- Not using Pipeline (harder to reproduce, easier to leak data)
